# R3-00 · Onramp express + Prueba de Nivel — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea C · *IA Aplicada* · Semana 0–1 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Manejar variables, listas y **comprensiones** sobre datos reales.
2. Escribir **funciones** que encapsulan reglas.
3. Leer **JSON** y aplicar el patrón **en vivo o caché** con APIs.
4. Aprobar la **Prueba de Nivel** para entrar directo a R3-01.

**Competencia de salida:** tener listo el prerequisito de Python/datos para la rama de IA Aplicada.

**Dato real:** montos de compras públicas reales (ChileCompra) + respuestas tipo API.

**Entregable:** que las **5 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd
import json, urllib.request

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Cargamos datos reales como base (arriba). Vamos por partes.

## 1. Variables, listas y comprensiones

En Python casi todo es **recorrer una colección**. Tomamos los montos reales de compras públicas como una lista y los resumimos sin pandas.

### ✍️ Ejercicio 1 — Resume la lista de montos

In [ ]:
montos = df["monto_total"].tolist()
total = sum(montos)
sobre_un_millon = len([m for m in montos if m > 1_000_000])
print("total:", total, "| órdenes > $1M:", sobre_un_millon)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert abs(total - df["monto_total"].sum()) < 1e-6
    assert sobre_un_millon == int((df["monto_total"] > 1_000_000).sum())
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'sum(montos) para el total; [m for m in montos if m > 1_000_000] y su len().')
    print("   Detalle:", e)

## 2. Funciones: empaquetar una decisión

Una **función** encapsula una regla reutilizable. Clasifiquemos cada compra por tramo de monto.

### ✍️ Ejercicio 2 — Escribe `clasificar`

In [ ]:
def clasificar(monto):
    if monto < 100_000:
        return "bajo"
    elif monto < 1_000_000:
        return "medio"
    return "alto"

print(clasificar(50_000), clasificar(500_000), clasificar(2_000_000))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert clasificar(50_000) == "bajo"
    assert clasificar(500_000) == "medio"
    assert clasificar(2_000_000) == "alto"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Usa if/elif/else comparando monto contra 100_000 y 1_000_000.')
    print("   Detalle:", e)

## 3. JSON: el idioma de las APIs

Las APIs devuelven **JSON**, que en Python se convierte en diccionarios y listas. Practiquemos extraer datos de una respuesta típica.

### ✍️ Ejercicio 3 — Parsea el JSON y suma los ítems

In [ ]:
texto = '{"orden": 123, "items": [{"nombre": "pan", "monto": 500}, {"nombre": "leche", "monto": 800}]}'
data = json.loads(texto)
monto_items = sum(it["monto"] for it in data["items"])
print("orden:", data["orden"], "| total ítems:", monto_items)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert data["orden"] == 123
    assert monto_items == 1300
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "json.loads(texto) da un dict; recorre data['items'] sumando it['monto'].")
    print("   Detalle:", e)

## 4. En vivo o caché: APIs que no te dejan tirado

Una API real puede fallar (sin red, caída, límite). El patrón **en vivo o caché**: intenta la llamada y, si falla, usa una copia. Así tu notebook siempre corre.

### ✍️ Ejercicio 4 — Escribe `cargar_o_cache`

In [ ]:
def cargar_o_cache(url, fallback):
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return json.load(r)
    except Exception:
        return fallback

datos = cargar_o_cache("http://no-existe.invalid/x.json", {"fuente": "cache", "n": 3})
print(datos)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert datos["fuente"] == "cache"
    assert callable(cargar_o_cache)
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'En el except, return fallback. La URL inválida fuerza el camino de caché.')
    print("   Detalle:", e)

## 5. Prueba de Nivel (PN) — ¿puedes saltar directo a R3-01?

Si resuelves esto **sin ayuda**, dominas el prerequisito (Python + colecciones + funciones) y puedes entrar directo a *Introducción al NLP*. Es el **corte** de entrada por la vía (b).

### ✍️ Ejercicio 5 — Resuelve las tres tareas de corte

In [ ]:
def filtra_mayores(lista, umbral):
    return [x for x in lista if x > umbral]

def cuenta_palabras(texto):
    return len(texto.split())

def gasto_por_clave(registros):
    out = {}
    for r in registros:
        out[r["region"]] = out.get(r["region"], 0) + r["monto"]
    return out

print(filtra_mayores([1, 5, 3, 8], 4), cuenta_palabras("compras públicas del estado"),
      gasto_por_clave([{"region": "A", "monto": 100}, {"region": "A", "monto": 50}]))

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert filtra_mayores([1, 5, 3, 8], 4) == [5, 8]
    assert cuenta_palabras("compras públicas del estado") == 4
    assert gasto_por_clave([{"region": "A", "monto": 100}, {"region": "A", "monto": 50}]) == {"A": 150}
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Comprensión con if; texto.split() y su len; acumula en un dict con .get(clave, 0).')
    print("   Detalle:", e)

## Cierre

- Recorrer colecciones, escribir **funciones** y manejar **JSON** es el 80% de lo que usarás en R3.
- El patrón **en vivo o caché** hace tus notebooks robustos cuando dependes de APIs.
- Si aprobaste la **Prueba de Nivel**, puedes saltar directo a **R3-01 · Introducción al NLP**.

> *En R3 usaremos un LLM (Gemini) detrás de `get_llm()`; aquí solo sentamos las bases de Python y datos.*